# Step 06b — LLM Classification (benchmark arm)

**Input:** the FROZEN condensed buildings file (`config.VALIDATION_BUILDINGS_FILE`)
**Output:** `data/validation/06b_llm_predictions_{arm}_run{n}.parquet`

This notebook classifies the **1,391 annotated buildings** with the LLM so that
notebook 10 can score it against the same ground truth, through the same code, as the
rule engine.

## Why this reads the frozen file and not notebook 05's output

Notebook 05 ends with `gdf['gml_id'] = gdf.index` — the id is a **positional row
index**, assigned after a label filter and after a concat whose length depends on a
spatial join. Byte-identical code has produced 578,080 / 574,435 / 567,961 rows on
different runs, so `gml_id` 236928 is a *different building* every time.

Measured against the annotation workbook:

| classifier input | ids that resolve | volume agreement |
|---|---|---|
| regenerated notebook 05 output | 1,312 / 1,391 (94.3%) | **0 of 1,312** |
| `VALIDATION_BUILDINGS_FILE` (frozen) | 1,391 / 1,391 | **100.00%** |

A regenerated file joins cleanly and compares unrelated buildings. That is why running
the same preprocessing code on both arms does **not** give the two arms the same data —
reading the same *file* does. The rule engine's published numbers were obtained exactly
this way: notebook 10 never opens `06_classified_buildings.gpkg`, it re-runs
`classify_building` in-process over this same frozen file. The arms share the function,
never a file.

## Two evidence arms

| `ARM` | fields sent | role |
|---|---|---|
| **`full`** | all 16, names included | **THE comparison.** Which classifier should be deployed. |
| `blind` | 13 of 16 — no `osm_names`, `website`, `email` | diagnostic: what the names are worth |

`full` is the headline arm. Giving the model business names is the entire reason to
use an LLM here — it knows what "Deutsche Bank", "Ernsting's family" or "Ilseder
Landkrug" are, and no rule table can encode that. `rule_utils` says names are
*"deliberately not used … the accepted limitation being measured here"*: that is the
rule engine's limitation, which the comparison should expose, not a handicap the LLM
should adopt to make the contest even.

`blind` withholds the same three fields the rule engine cannot use. It is not the
fair comparison — it is the **ablation**. `full − blind` is precisely what
business-name world knowledge buys, and that difference is the argument for the LLM.

## Cost

One building per request, stateless, and **one request at a time**
(`LLM_MAX_WORKERS = 1`) — which removes every concurrency-shaped failure mode. The
endpoint's rate limit is undocumented and the retry path treats HTTP 429 like a network
error with a 2/4/6 s backoff, so a parallel run that trips it retries into the same wall
three times and loses the row.

`SCOPE` controls how many of the 1,391 annotated buildings are sent:

| `SCOPE` | rows | what it buys |
|---|---|---|
| `'scoreable'` *(default)* | **885** | every metric in notebook 10, in full |
| `'all'` | 1,391 | the same metrics, plus a wider self-consistency diagnostic |

The 506 extra rows have no ground-truth answer on either dimension — the validator was
uncertain (337), never reached them (159), or flagged them wrong without typing a
correction (10). Notebook 10 drops them for **every** arm, so no precision, recall or
accuracy figure changes between the two settings. They do still carry a pre-filled value
from the earlier LLM run, which is the one thing classifying them adds.

| per call | 885 rows | 1,391 rows |
|---|---|---|
| 15 s | 3.7 h | 5.8 h |
| 30 s | 7.4 h | 11.6 h |
| 60 s | 14.8 h | 23.2 h |

**Don't plan on any of those.** Step 2 prints the measured `s/row` and a live ETA after
every chunk — the first chunk gives the real number within ~10–25 minutes. Use that.

It checkpoints every 25 rows, so an interruption costs at most one chunk and re-running
resumes where it stopped. Safe to leave overnight.

Errors never enter the checkpoint, so re-running this notebook *is* the retry mechanism.
Drive errors to zero — the rule engine has no exclusions (882/882 and 874/874), and any
LLM row left failing shrinks the LLM's denominator and breaks comparability.

In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import (
    VALIDATION_BUILDINGS_FILE, VALIDATION_GROUND_TRUTH,
    llm_validation_checkpoint, llm_validation_errors,
    LLM_MODEL, LLM_MAX_WORKERS, LLM_CHUNK_SIZE, TARGET_MID_LABELS,
)
from llm_utils import SYSTEM_PROMPT, row_to_llm_input, predict_row

import time
import pandas as pd
import geopandas as gpd
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Parameters ────────────────────────────────────────────────────────────────
ARM = 'full'    # 'full' = THE headline arm | 'blind' = the names-withheld diagnostic
                #
                # 'full' gives the model every attribute, business names included.
                # That is the point of using an LLM at all: it carries world
                # knowledge about "Deutsche Bank" or "Ernsting's family" that no
                # rule table can encode. Withholding names to match the rule
                # engine's inputs measures an academic question nobody needs
                # answered; the deployable question is which classifier to run,
                # and the rule engine's blindness to names is the limitation being
                # demonstrated, not an unfairness to be corrected.
                #
                # 'blind' remains worth running as a DIAGNOSTIC: full - blind is
                # exactly what business-name world knowledge is worth, and that
                # difference is the argument for the LLM over the rules.
RUN = 1         # repeat index; the LLM is not deterministic and nothing downstream
                # measures that, so >1 run is how the arm gets an error bar

SCOPE = 'scoreable'   # 'scoreable' (885 rows) | 'all' (1,391 rows)
# 'scoreable' classifies only the buildings that have a ground-truth answer on at
#   least one dimension. Every metric in notebook 10 is identical either way,
#   because the scorer's denominators come from activities_scoreable /
#   bosserhof_scoreable and never included the other 506 rows.
# 'all' additionally classifies the 506 buildings the validator marked uncertain
#   (337), never reached (159), or flagged wrong without correcting (10). They
#   cannot be scored, but they DO have a pre-filled value from the earlier LLM
#   run, so classifying them widens the Step-10 self-consistency diagnostic from
#   885 to 1,391 rows. That is the only thing the extra 36% of spend buys.

CHECKPOINT = llm_validation_checkpoint(ARM, RUN)
ERRORS     = llm_validation_errors(ARM, RUN)

for path, what in [(VALIDATION_BUILDINGS_FILE, 'the frozen benchmark buildings'),
                   (VALIDATION_GROUND_TRUTH, 'notebook 09 output')]:
    if not path.exists():
        raise FileNotFoundError(f'{path}\n  missing: {what}')

print(f'arm={ARM}  run={RUN}  model={LLM_MODEL}  workers={LLM_MAX_WORKERS}')
print(f'checkpoint -> {CHECKPOINT.name}')
print(f'errors     -> {ERRORS.name}')
print(f'\nprompt: {len(SYSTEM_PROMPT):,} chars')

arm=full  run=1  model=gpt-oss-120b  workers=1
checkpoint -> 06b_llm_predictions_full_run1.parquet
errors     -> 06b_llm_errors_full_run1.parquet

prompt: 3,179 chars


---
## Step 1 — Load the annotated buildings, and only those

The filter runs **before** the sentence-building `.apply`: `read_file` loads all 578,080
rows and rendering a prompt for every one of them wastes minutes for nothing.

The assertion on 1,391 is the real check that this is the right input file. A
regenerated file would drop ~79 ids here rather than silently classifying the wrong
buildings.

In [2]:
truth = pd.read_parquet(VALIDATION_GROUND_TRUTH)
truth['gml_id'] = truth['gml_id'].astype(str)
all_ids = set(truth['gml_id'])

# Classify ONLY the rows that can actually be scored on at least one dimension.
# The other 506 have no ground truth on EITHER dimension — the validator was
# uncertain (337), never reached them (159), or flagged them wrong without typing
# a correction (10) — so notebook 10 drops them for every arm regardless. Paying
# for a prediction on them buys nothing.
#
# This does NOT affect comparability: the scorer's denominators come from
# activities_scoreable / bosserhof_scoreable, which never included these rows.
can_score = truth['activities_scoreable'] | truth['bosserhof_scoreable']
if SCOPE == 'scoreable':
    ids = set(truth.loc[can_score, 'gml_id'])
elif SCOPE == 'all':
    ids = set(all_ids)
else:
    raise ValueError(f"SCOPE must be 'scoreable' or 'all', got {SCOPE!r}")

print(f'{len(all_ids):,} annotated buildings')
print(f'{int(can_score.sum()):,} scoreable on at least one dimension')
print(f'{int((~can_score).sum()):,} unscoreable on both '
      f'(uncertain / never reviewed / flagged but not corrected)')
print(f'\nSCOPE={SCOPE!r} -> classifying {len(ids):,}')
if SCOPE == 'all':
    print('  the extra rows cannot be scored; they only widen the Step 10 '
          'self-consistency diagnostic')

pois = gpd.read_file(VALIDATION_BUILDINGS_FILE)
print(f'\n{len(pois):,} buildings in {VALIDATION_BUILDINGS_FILE.name}')

pois['gml_id'] = pois['gml_id'].astype(str)
present_all = int(pois['gml_id'].isin(all_ids).sum())
assert present_all == len(all_ids), (
    f'expected all {len(all_ids):,} annotated ids in the source, found {present_all:,}. '
    'gml_id is a per-run positional index — this is almost certainly a REGENERATED '
    'condensed file rather than the frozen one the workbook indexes into. Classifying it '
    'would compare different buildings while every join looks healthy.')

pois = pois[pois['gml_id'].isin(ids)].copy()
assert len(pois) == len(ids), f'expected {len(ids):,} scoreable buildings, matched {len(pois):,}'
assert pois['gml_id'].is_unique, 'gml_id not unique in the source'

# Volume agreement proves the ids point at the SAME buildings, not merely at ids that
# happen to exist. This is the check that fails loudly on a regenerated file.
chk = truth.merge(pois[['gml_id', 'volume_m3']], on='gml_id', how='inner',
                  suffixes=('_truth', '_src'))
rel = ((chk['volume_m3_truth'] - chk['volume_m3_src']).abs()
       / chk['volume_m3_truth'].abs().clip(lower=1e-9))
share = float((rel < 1e-3).mean())
print(f'volume agreement with the workbook: {share:.2%}')
assert share > 0.95, (
    f'only {share:.1%} of volumes agree — the ids resolve but describe different '
    'buildings. Refusing to spend API calls on this input.')

pois['sentence'] = pois.apply(lambda r: row_to_llm_input(r, fields=ARM), axis=1)
empty = int((pois['sentence'].str.len() == 0).sum())
assert empty == 0, f'{empty} buildings rendered an EMPTY prompt — the model would be asked nothing'

print(f'\nprompts built (fields={ARM}); median length '
      f'{int(pois["sentence"].str.len().median())} chars')
print('\nexample:\n')
print(pois['sentence'].iloc[0])

1,391 annotated buildings
885 scoreable on at least one dimension
506 unscoreable on both (uncertain / never reviewed / flagged but not corrected)

SCOPE='scoreable' -> classifying 885


C:\Users\Mayur Patel\anaconda3\envs\GNNs\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()



578,080 buildings in condensed_buildings_with_pois.gpkg


volume agreement with the workbook: 100.00%

prompts built (fields=full); median length 236 chars

example:

general_building_context: building_label=Buildings for business or commerce | osm_landuse_class=industrial | gfk_class=Gebäude | alkis_landuse=industrial


---
## Step 2 — Classify, checkpointing as we go

`append_parquet` rewrites the accumulated file on every flush, so an interrupted run
loses at most one chunk. Successes go to the checkpoint, failures to a separate errors
file and **never** to the checkpoint — which is what makes re-running this cell the
retry mechanism.

Every prediction is stamped with `src_file` and `src_volume_m3`. Notebook 10 asserts on
them; without the stamp its identity guard compares a value it re-joined from the source
against itself, which is bit-identical by construction and proves nothing.

**Fail fast:** if the first chunk errors above 5%, the run aborts. A systematic fault
(bad token, changed response shape, wrong argument arity) then costs 25 calls instead of
1,391 and most of a day. Run `scripts/llm_smoke_test.py --n 2` first — it catches the
same faults in two calls.

`ThreadPoolExecutor` is kept at `max_workers=1` rather than removed, so raising the
concurrency later needs a config change and no code change.

In [3]:
done_ids = set()
if CHECKPOINT.exists():
    done_ids = set(pd.read_parquet(CHECKPOINT)['gml_id'].astype(str))
    print(f'Resuming: {len(done_ids):,} already done')

todo = pois[~pois['gml_id'].isin(done_ids)].copy()
print(f'Remaining to classify: {len(todo):,}')


def process_chunk(chunk_df):
    results = []
    with ThreadPoolExecutor(max_workers=LLM_MAX_WORKERS) as executor:
        futures = [
            executor.submit(predict_row, row['gml_id'], row['sentence'], ARM,
                            VALIDATION_BUILDINGS_FILE.name, row['volume_m3'])
            for _, row in chunk_df.iterrows()
        ]
        for future in as_completed(futures):
            results.append(future.result())
    return results


def append_parquet(path, new_rows):
    new_df = pd.DataFrame(new_rows)
    if path.exists():
        combined = pd.concat([pd.read_parquet(path), new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset='gml_id', keep='last')
    else:
        combined = new_df
    path.parent.mkdir(parents=True, exist_ok=True)
    combined.to_parquet(path, index=False)


chunks = [todo.iloc[i:i + LLM_CHUNK_SIZE] for i in range(0, len(todo), LLM_CHUNK_SIZE)]
total_done = len(done_ids)
t_start = time.time()
n_called = 0

for i, chunk in enumerate(chunks):
    t_chunk = time.time()
    results = process_chunk(chunk)
    good   = [r for r in results if r['error'] is None]
    errors = [r for r in results if r['error'] is not None]

    if good:   append_parquet(CHECKPOINT, good)
    if errors: append_parquet(ERRORS, errors)

    total_done += len(results)
    n_called += len(results)

    # Measured rate, not an estimate. After the first chunk this is the real
    # answer to "how long will this take" — replace any guess with it.
    sec_per_row = (time.time() - t_start) / max(n_called, 1)
    remaining = len(todo) - n_called
    eta_h = remaining * sec_per_row / 3600
    print(f'Chunk {i+1}/{len(chunks)} | done={total_done}/{len(pois)} | '
          f'errors={len(errors)} | {sec_per_row:5.1f} s/row | '
          f'chunk {(time.time() - t_chunk) / 60:4.1f} min | ETA {eta_h:5.2f} h')

    if i == 0 and len(errors) > 0.05 * len(chunk):
        for e in errors[:3]:
            print(f'    {e["error"]}')
        raise RuntimeError(
            f'{len(errors)}/{len(chunk)} failed in the FIRST chunk. Aborting rather than '
            'burning hours on a systematic fault. Check the token, the endpoint and the '
            'response shape, then re-run — completed rows are already checkpointed.')

print('\nClassification pass complete.')

Resuming: 884 already done
Remaining to classify: 1


Chunk 1/1 | done=885/885 | errors=0 |  28.7 s/row | chunk  0.5 min | ETA  0.00 h

Classification pass complete.


---
## Step 3 — Coverage

The rule engine excludes nothing. Any LLM row still failing here shrinks the LLM's
denominator, and two arms scored over different building sets are not comparable.

Re-run Step 2 until this reports zero. If a row fails repeatedly, report the residual
count alongside the metrics rather than quietly scoring 1,388 buildings against the rule
engine's 1,391.

In [4]:
n_done = 0
if CHECKPOINT.exists():
    ck = pd.read_parquet(CHECKPOINT)
    n_done = len(ck)
    assert ck['gml_id'].is_unique, 'duplicate gml_id in the checkpoint'
    assert (ck['src_file'] == VALIDATION_BUILDINGS_FILE.name).all(), \
        'checkpoint mixes predictions derived from different source files'

n_err = 0
if ERRORS.exists():
    err = pd.read_parquet(ERRORS)
    err = err[~err['gml_id'].astype(str).isin(ck['gml_id'].astype(str))] if n_done else err
    n_err = len(err)

print(f'classified : {n_done:,} / {len(pois):,}')
print(f'outstanding: {n_err:,}')

if n_err:
    print('\nFailure modes:')
    print(err['error'].str.slice(0, 90).value_counts().head(10).to_string())
    print('\nRe-run Step 2 — errors are not checkpointed, so it retries exactly these.')
else:
    print('\nFull coverage. Comparable with the rule engine denominator.')

if n_done:
    print('\nLabel distribution:')
    print(ck['mid_labels'].explode().value_counts().to_string())
    print(f'\nnull bosserhof_class: {int(ck["bosserhof_class"].isna().sum()):,}')

classified : 885 / 885
outstanding: 0

Full coverage. Comparable with the rule engine denominator.

Label distribution:
mid_labels
work                804
errands             219
leisure             182
retail_non_daily    177
meetup               64
retail_daily         52
school               49
business             32
lessons              28
university           23
sports               18
childcare            17

null bosserhof_class: 50


---
## Done

Score it with **notebook 10**, which discovers every `06b_llm_predictions_*.parquet`
and scores each as its own arm against the same ground truth, through the same
`validation_utils` calls, as the rule engine.

To produce the other arms, set `ARM`/`RUN` at the top and re-run:

| `ARM` | `RUN` | purpose |
|---|---|---|
| `blind` | 1 | primary method comparison |
| `full` | 1 | value of business names |
| `blind` | 2, 3 | run-to-run variance (optional) |

Each arm is 885 calls. Start with `blind` run 1 — it is the run that answers
"LLM vs rules".

---

### Production run (not part of the benchmark — leave unrun)

Classifying the full `CONDENSED_BUILDINGS_FILE` writes `LLM_CHECKPOINT_FILE`, feeds
notebook 06c, and produces zone-level results comparable to `08_final_results.gpkg`.

At 15–45 s/call and 4 workers, 578,080 buildings is **25–75 days of wall clock**. It
answers no question the 1,391-row benchmark does not already answer. The cell below is
deliberately left unexecuted.

In [5]:
# PRODUCTION RUN — 578,080 calls, 25-75 days. Do not run casually.
#
# Uses LLM_CHECKPOINT_FILE, deliberately separate from the benchmark checkpoints: the
# frozen and regenerated files draw gml_ids from the same small-integer namespace while
# describing different buildings, so a shared checkpoint would interleave predictions for
# unrelated buildings and nothing downstream could detect it.
#
# from config import CONDENSED_BUILDINGS_FILE, LLM_CHECKPOINT_FILE, LLM_ERRORS_FILE
# prod = gpd.read_file(CONDENSED_BUILDINGS_FILE)
# prod['sentence'] = prod.apply(lambda r: row_to_llm_input(r, fields='full'), axis=1)
# ... same chunk loop, writing LLM_CHECKPOINT_FILE / LLM_ERRORS_FILE ...
pass